## 1. Mục tiêu phần UI

Mục tiêu của phần UI là đóng gói pipeline RAG của nhóm thành một ứng dụng web thân thiện, cho phép người dùng:

- Upload file PDF.
- Bấm nút xử lý PDF.
- Hỏi đáp dựa trên nội dung tài liệu.
- Xem lịch sử hội thoại.
- Xóa lịch sử chat khi cần.

Thay vì phải sửa code mỗi lần hỏi, người dùng có thể thao tác trực tiếp trên giao diện Streamlit.

## 2. Tổng quan pipeline

Luồng xử lý của hệ thống:

```text
PDF upload
→ Đọc nội dung PDF bằng pypdf
→ Chunking
→ Embedding bằng bge-m3 qua Ollama
→ Lưu vào ChromaDB
→ Retrieve top-k chunks liên quan
→ Ghép context + câu hỏi vào prompt
→ Gọi LLM Vicuna qua Ollama
→ Hiển thị câu trả lời trên Streamlit
```

Trong đó, phần em phụ trách là giao diện Streamlit và phần kết nối UI với các hàm RAG của nhóm.

## 3. Mô tả các thành phần chính trong source code

### `load_pdf(file_path)`

Đọc file PDF bằng `pypdf.PdfReader`, duyệt từng trang và trích xuất text. Nếu trang không có text, hàm thay bằng chuỗi rỗng để tránh lỗi.

### `chunk_text(text, chunk_size=1000, chunk_overlap=200)`

Cắt văn bản dài thành nhiều chunk nhỏ. `chunk_size=1000` giúp mỗi đoạn vừa đủ ngữ cảnh, còn `chunk_overlap=200` giúp giữ thông tin ở ranh giới giữa hai chunk.

### `create_vector_db(chunks)`

Tạo ChromaDB collection và sử dụng `bge-m3` qua Ollama để embedding từng chunk rồi lưu vào vector database.

### `retrieve_context(query, collection, n_results=4)`

Nhận câu hỏi của người dùng và truy vấn ra các chunk liên quan nhất từ ChromaDB. `n_results=4` nghĩa là lấy tối đa 4 đoạn liên quan nhất.

### `generate_answer(query, retrieved_chunks)`

Ghép các chunk liên quan thành context, tạo prompt nghiêm ngặt và gọi LLM `vicuna:7b-v1.5-q5_1` để sinh câu trả lời. Prompt yêu cầu model chỉ trả lời dựa trên tài liệu và không bịa thông tin.

### `process_pdf(uploaded_file)`

Hàm trung gian cho Streamlit. File upload từ `st.file_uploader` nằm trong bộ nhớ, nên hàm lưu thành file tạm rồi gọi pipeline đọc PDF, chunking và tạo vector database.

### `rag(question, collection)`

Hàm RAG chính cho UI: nhận câu hỏi, retrieve context và gọi LLM để trả lời.

## 4. Mô tả giao diện Streamlit

Giao diện gồm hai khu vực chính:

### Sidebar

- Upload file PDF.
- Nút **Xử lý PDF**.
- Hiển thị tên tài liệu đã upload.
- Nút **Xóa lịch sử chat**.

### Main chat UI

- Hiển thị lịch sử hội thoại bằng `st.chat_message`.
- Nhận câu hỏi bằng `st.chat_input`.
- Nếu chưa xử lý PDF, ô chat bị khóa để tránh lỗi.
- Khi người dùng hỏi, app gọi hàm `rag()` để trả lời dựa trên tài liệu.

## 5. Quản lý trạng thái bằng `st.session_state`

Streamlit chạy lại toàn bộ file Python sau mỗi lần người dùng tương tác, vì vậy app sử dụng `st.session_state` để lưu:

- `collection`: vector database của PDF đang xử lý.
- `pdf_name`: tên file PDF.
- `chat_history`: lịch sử hội thoại.
- `n_chunks`: số chunk đã tạo.
- `n_pages`: số trang của PDF.

Nhờ đó, dữ liệu không bị reset sau mỗi lần người dùng nhập câu hỏi hoặc bấm nút.

## 6. Cell cài đặt thư viện Python

Chạy cell này đầu tiên trong Google Colab.

In [ ]:
!pip install -q streamlit pypdf chromadb ollama

## 7. Cài Ollama trên Google Colab

Colab không có Ollama sẵn nên cần cài thủ công.

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

## 8. Chạy Ollama server và kiểm tra model

Cell này chạy Ollama server nền. Sau đó dùng `ollama list` để kiểm tra model đã có chưa.

In [ ]:
!nohup ollama serve > ollama.log 2>&1 &
!sleep 15
!ollama list

## 9. Tải model nếu chưa có

Nếu `ollama list` chưa có `bge-m3` và `vicuna:7b-v1.5-q5_1`, chạy cell này.

Lưu ý: `vicuna:7b-v1.5-q5_1` khá nặng. Nếu Colab chạy chậm, có thể đổi sang `qwen2.5:3b` trong source code.

In [ ]:
!ollama pull bge-m3
!ollama pull vicuna:7b-v1.5-q5_1
!ollama list

## 10. Tạo file app Streamlit

Cell này ghi source code thành file `chatbot_app_native.py`.  
Khi chạy app Streamlit trên Colab vẫn cần tạo file `.py` tạm thời để lệnh `streamlit run` sử dụng.

In [ ]:
%%writefile chatbot_app_native.py
import streamlit as st
import tempfile
import os
import time

import pypdf
import chromadb
import ollama


# 1. Đọc PDF -> tạo chuỗi văn bản
def load_pdf(file_path: str) -> str:
    """
    Đọc file PDF từ đường dẫn cho trước và trích xuất toàn bộ văn bản.

    Args:
        file_path (str): Đường dẫn tới file PDF cần đọc.

    Returns:
        str: Chuỗi văn bản tổng hợp từ tất cả các trang của PDF.
    """
    # Khởi tạo đối tượng PdfReader từ thư viện pypdf
    reader = pypdf.PdfReader(file_path)

    pages_text = []

    # Duyệt qua từng trang của tài liệu
    for page in reader.pages:
        # Trích xuất text từ trang hiện tại
        text = page.extract_text()

        # Phòng thủ lỗi: Nếu trang chỉ có ảnh hoặc lỗi không lấy được text,
        # extract_text() sẽ trả về None. Ta thay thế bằng chuỗi rỗng "".
        if text is None:
            text = ""

        pages_text.append(text)

    # Gộp toàn bộ văn bản của các trang lại với nhau, phân tách bằng dấu xuống dòng
    full_text = "\n".join(pages_text)
    return full_text


# 2. Chunking
def chunk_text(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> list:
    """
    Cắt một chuỗi văn bản lớn thành các đoạn nhỏ hơn (chunks) dựa trên
    kích thước chunk_size và độ trùng lặp chunk_overlap.

    Args:
        text (str): Chuỗi văn bản đầu vào khổng lồ.
        chunk_size (int): Số lượng ký tự tối đa của một chunk.
        chunk_overlap (int): Số lượng ký tự trùng lặp gối đầu giữa các chunk liên tiếp.

    Returns:
        list: Danh sách các chuỗi văn bản đã được cắt nhỏ.
    """
    chunks = []
    words = text.split()  # Tách văn bản thành danh sách các từ dựa trên khoảng trắng
    full_text_cleaned = " ".join(words)  # Chuẩn hóa văn bản loại bỏ khoảng trắng thừa

    start_idx = 0
    text_length = len(full_text_cleaned)

    # Vòng lặp dịch chuyển cửa sổ cắt (Sliding Window)
    while start_idx < text_length:
        # Xác định vị trí kết thúc dự kiến của chunk hiện tại
        end_idx = start_idx + chunk_size

        # Lấy ra đoạn văn bản trong khung cửa sổ
        chunk = full_text_cleaned[start_idx:end_idx]

        if chunk.strip():
            chunks.append(chunk)

        # Dịch chuyển vị trí bắt đầu cho chunk tiếp theo.
        # Khoảng dịch chuyển bằng (Kích thước chunk - Độ trùng lặp)
        start_idx += (chunk_size - chunk_overlap)

    return chunks


# 3. Vector hóa và lưu trữ vào Vector Database
def create_vector_db(chunks: list, collection_name: str = "rag_collection") -> chromadb.Collection:
    """
    Khởi tạo ChromaDB, tạo embedding cho các đoạn chunks thông qua Ollama
    và lưu chúng vào một Collection.

    Args:
        chunks (list): Danh sách các đoạn văn bản đã được cắt nhỏ.
        collection_name (str): Tên của nhóm lưu trữ dữ liệu trong ChromaDB.

    Returns:
        chromadb.Collection: Đối tượng Collection đã chứa dữ liệu và sẵn sàng truy vấn.
    """
    # 1. Khởi tạo Client của ChromaDB
    chroma_client = chromadb.Client()

    # 2. Định nghĩa hàm tự tạo Embedding bằng mô hình bge-m3 thông qua Ollama.
    class OllamaEmbeddingFunction:

        def _embed(self, texts):
            embeddings = []

            for text in texts:
                response = ollama.embeddings(
                    model="bge-m3",
                    prompt=text
                )
                embeddings.append(response["embedding"])

            return embeddings

        def __call__(self, input):
            return self._embed(input)

        def embed_documents(self, input):
            return self._embed(input)

        def embed_query(self, input):
            if isinstance(input, str):
                input = [input]

            return self._embed(input)

        def name(self):
            return "ollama-bge-m3"

    embedding_function = OllamaEmbeddingFunction()

    # 3. Tạo hoặc lấy lại một Collection
    collection = chroma_client.get_or_create_collection(
        name=collection_name,
        embedding_function=embedding_function
    )

    # 4. Chuẩn bị dữ liệu để nạp vào Database
    ids = [f"chunk_{i}" for i in range(len(chunks))]

    # Nạp dữ liệu vào Collection.
    collection.add(
        documents=chunks,
        ids=ids
    )

    return collection


# 4. Truy vấn
def retrieve_context(query: str, collection: chromadb.Collection, n_results: int = 4) -> list:
    """
    Nhận câu hỏi từ người dùng, tự động vector hóa câu hỏi và truy vấn
    ra các đoạn văn bản liên quan nhất từ ChromaDB.

    Args:
        query (str): Câu hỏi thô từ người dùng nhập vào.
        collection (chromadb.Collection): Đối tượng kho lưu trữ dữ liệu đã tạo.
        n_results (int): Số lượng đoạn văn bản liên quan nhất cần lấy ra.

    Returns:
        list: Danh sách các chuỗi văn bản có ngữ nghĩa sát với câu hỏi nhất.
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]

    return retrieved_chunks


# 5. Connect LLM sinh câu trả lời
def generate_answer(query: str, retrieved_chunks: list, model_name: str = "vicuna:7b-v1.5-q5_1") -> str:
    """
    Kết hợp câu hỏi và ngữ cảnh tìm được thành một Prompt nghiêm ngặt,
    sau đó gọi mô hình LLM để sinh câu trả lời chính xác.

    Args:
        query (str): Câu hỏi của người dùng.
        retrieved_chunks (list): Danh sách các đoạn văn bản liên quan.
        model_name (str): Tên mô hình LLM đang chạy trên Ollama.

    Returns:
        str: Câu trả lời cuối cùng từ chatbot.
    """
    # 1. Gộp các đoạn chunks lại thành một khối văn bản duy nhất để làm ngữ cảnh
    context = "\n---\n".join(retrieved_chunks)

    # 2. Xây dựng cấu trúc Prompt nghiêm ngặt
    prompt = f"""Bạn là một trợ lý hỏi đáp tài liệu học tập thông minh và trung thực.
Hãy sử dụng các đoạn ngữ cảnh (Context) được cung cấp dưới đây để trả lời câu hỏi (Question).

---
NGỮ CẢNH (CONTEXT):
{context}
---

CÂU HỎI (QUESTION):
{query}

---
YÊU CẦU:
- Trả lời ngắn gọn, rõ ràng, đi thẳng vào vấn đề.
- Chỉ dựa vào thông tin có trong phần NGỮ CẢNH phía trên.
- Nếu thông tin trong NGỮ CẢNH không có hoặc không đủ để trả lời, hãy nói thẳng là "Tôi không biết câu trả lời dựa trên tài liệu đã cung cấp", TUYỆT ĐỐI KHÔNG ĐƯỢC BỊA ĐẶT thông tin.

TRẢ LỜI:"""

    # 3. Gọi mô hình LLM thông qua API của Ollama
    response = ollama.generate(
        model=model_name,
        prompt=prompt,
        options={
            "temperature": 0.0
        }
    )

    answer = response["response"]

    return answer


# ======================================================
# PHẦN BỔ SUNG CHO STREAMLIT UI
# ======================================================

LLM_MODEL = "vicuna:7b-v1.5-q5_1"

# Nếu Colab quá chậm, có thể đổi thành model nhẹ hơn:
# LLM_MODEL = "qwen2.5:3b"


def process_pdf(uploaded_file):
    """
    Xử lý file PDF từ Streamlit: đọc text, chia nhỏ (chunk) và tạo Vector DB.

    Args:
        uploaded_file: File PDF tải lên từ Streamlit.

    Returns:
        tuple: (collection, n_chunks, n_pages)
    """
    # Lưu file tạm để pypdf có thể đọc
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(uploaded_file.getvalue())
        temp_path = tmp.name

    try:
        # Trích xuất text và đếm số trang
        full_text = load_pdf(temp_path)
        reader = pypdf.PdfReader(temp_path)
        n_pages = len(reader.pages)
    finally:
        # Đảm bảo luôn xóa file tạm sau khi đọc xong
        if os.path.exists(temp_path):
            os.unlink(temp_path)

    # Kiểm tra nếu PDF không trích xuất được text (ví dụ: file scan ảnh)
    if not full_text.strip():
        raise ValueError("Không trích xuất được nội dung từ PDF. Có thể file là PDF scan ảnh.")

    # Chia text thành các đoạn nhỏ (chunks)
    chunks = chunk_text(full_text)
    if len(chunks) == 0:
        raise ValueError("Không tạo được chunk từ tài liệu.")

    # Tạo Vector DB trong ChromaDB (dùng timestamp để đặt tên collection không trùng)
    collection_name = f"rag_collection_{int(time.time() * 1000)}"
    collection = create_vector_db(chunks, collection_name=collection_name)

    return collection, len(chunks), n_pages


def rag(question, collection):
    """
    Hàm RAG chính cho UI:
    Nhận câu hỏi -> retrieve context -> generate answer.
    """

    top_k = min(4, collection.count())

    retrieved_chunks = retrieve_context(
        query=question,
        collection=collection,
        n_results=top_k
    )

    answer = generate_answer(
        query=question,
        retrieved_chunks=retrieved_chunks,
        model_name=LLM_MODEL
    )

    return answer


# ======================================================
# STREAMLIT UI
# ======================================================

# Cấu hình trang
st.set_page_config(
    page_title="PDF RAG Chatbot",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("🤖 PDF RAG Assistant")
st.write("Upload tài liệu PDF và hỏi đáp trực tiếp dựa trên nội dung tài liệu.")


# ======================================================
# SESSION STATE
# ======================================================

for k, v in {
    "collection": None,
    "pdf_name": "",
    "chat_history": [],
    "n_chunks": 0,
    "n_pages": 0
}.items():
    st.session_state.setdefault(k, v)


# ======================================================
# SIDEBAR
# ======================================================

# Sidebar: upload PDF và nút điều khiển
with st.sidebar:
    st.subheader("Upload tài liệu")
    f = st.file_uploader("Chọn file PDF", type="pdf")

    if f and st.button("Xử lý PDF", use_container_width=True):
        with st.spinner("Đang xử lý..."):
            try:
                # Hàm process_pdf trả về tuple (collection, n_chunks, n_pages)
                # Ta unpack và gán tương ứng vào session state
                collection, n_chunks, n_pages = process_pdf(f)
                st.session_state.collection = collection
                st.session_state.n_chunks = n_chunks
                st.session_state.n_pages = n_pages
                st.session_state.pdf_name = f.name
                st.session_state.chat_history = []
                st.success(f"{n_chunks} chunks")
            except Exception as e:
                st.error(f"Lỗi xử lý: {e}")

    st.info(f"📁 {st.session_state.pdf_name}" if st.session_state.pdf_name else "⚠️ Chưa có tài liệu")

    if st.button("Xóa lịch sử chat", use_container_width=True):
        st.session_state.chat_history = []
        st.rerun()


# ======================================================
# MAIN CHAT UI
# ======================================================

# Hiển thị lịch sử chat
for m in st.session_state.chat_history:
    with st.chat_message(m["role"]):
        st.write(m["content"])

# Ô nhập câu hỏi
if st.session_state.collection is None:
    st.info("Upload và xử lý PDF trước khi chat.")
    st.chat_input("Nhập câu hỏi...", disabled=True)
else:
    q = st.chat_input("Nhập câu hỏi của bạn...")
    if q:
        st.session_state.chat_history.append({"role": "user", "content": q})
        with st.chat_message("user"):
            st.write(q)

        with st.chat_message("assistant"):
            with st.spinner("Đang suy nghĩ..."):
                try:
                    ans = rag(q, st.session_state.collection)
                    st.write(ans)
                except Exception as e:
                    ans = f"Đã xảy ra lỗi: {e}"
                    st.error(ans)

        st.session_state.chat_history.append({"role": "assistant", "content": ans})
        st.rerun()

## 11. Kiểm tra source code

Nếu cell dưới không in lỗi, source code không bị lỗi cú pháp.

In [ ]:
!python -m py_compile chatbot_app_native.py
!ls -lh chatbot_app_native.py

## 12. Chạy Streamlit server

Chạy Streamlit trên port `8501`. Sau đó kiểm tra server có phản hồi chưa.

In [ ]:
!python -m streamlit run chatbot_app_native.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &
!sleep 10
!curl -I http://localhost:8501

## 13. Mở app bằng Localtunnel

Trước tiên lấy Endpoint IP. Khi mở link `.loca.lt`, nếu trang hỏi Endpoint IP thì dán IP này vào.

In [ ]:
!curl https://loca.lt/mytunnelpassword

Sau đó chạy Localtunnel. Cell này sẽ hiện link dạng `https://xxxxx.loca.lt`.

In [ ]:
!npx --yes localtunnel --port 8501

## 14. Cách test app

Sau khi mở link Streamlit:

1. Upload một file PDF ở sidebar.
2. Bấm **Xử lý PDF**.
3. Đợi app báo số chunks.
4. Nhập câu hỏi vào ô chat.
5. Kiểm tra chatbot trả lời dựa trên nội dung PDF.
6. Bấm **Xóa lịch sử chat** để kiểm tra reset hội thoại.